# IEEE-CIS Stage 1 Probability Calibration (Brier Score and ECE)

Verification of baseline Stage 1 probability calibration metrics reported in Section 4.1.1 of the manuscript.

## CELL 15 — Probability Calibration Check (Section 4.1.1)
Evaluates Brier score and Expected Calibration Error (ECE) on the held-out IEEE-CIS test partition (n = 59,054) across the full distribution and top-decile risk corridor.

In [15]:

# =============================================================================
# CELL 15 -- CALIBRATION + FLAT-COST ROBUSTNESS CHECK (illustrative only, design note #5)
# =============================================================================
y_test_arr = y_test.values
brier = brier_score_loss(y_test_arr, y_test_proba)

def ece_uniform(y_true, y_prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (y_prob > lo) & (y_prob <= hi) if lo > 0 else (y_prob >= lo) & (y_prob <= hi)
        if mask.sum() > 0:
            ece += abs(y_prob[mask].mean() - y_true[mask].mean()) * mask.mean()
    return ece

def ece_adaptive(y_true, y_prob, n_bins=10):
    order = np.argsort(y_prob)
    y_true_s, y_prob_s = y_true[order], y_prob[order]
    bins = np.array_split(np.arange(len(y_prob_s)), n_bins)
    ece = 0.0
    for b in bins:
        if len(b) > 0:
            ece += abs(y_prob_s[b].mean() - y_true_s[b].mean()) * (len(b) / len(y_prob_s))
    return ece

ece_u = ece_uniform(y_test_arr, y_test_proba)
ece_a = ece_adaptive(y_test_arr, y_test_proba)

top_decile_cut = np.quantile(y_test_proba, 0.90)
mask_top = y_test_proba >= top_decile_cut
brier_top = brier_score_loss(y_test_arr[mask_top], y_test_proba[mask_top])
ece_top = ece_uniform(y_test_arr[mask_top], y_test_proba[mask_top], n_bins=10)

print("--- Calibration ---")
print(f"Brier Score (full test set)         : {brier:.6f}")
print(f"ECE, uniform 10 bins (full test set): {ece_u:.6f}")
print(f"ECE, adaptive/quantile 10 bins       : {ece_a:.6f}")
print(f"Top-decile-only (predicted prob >= {top_decile_cut:.4f}, n={mask_top.sum()}):")
print(f"  Brier Score (top decile)  : {brier_top:.6f}")
print(f"  ECE, uniform (top decile) : {ece_top:.6f}")


--- Calibration ---
Brier Score (full test set)         : 0.030565
ECE, uniform 10 bins (full test set): 0.066342
ECE, adaptive/quantile 10 bins       : 0.066342
Top-decile-only (predicted prob >= 0.2501, n=5906):
  Brier Score (top decile)  : 0.218940
  ECE, uniform (top decile) : 0.283867
